In [1]:
# For Optical Character Recognition (OCR) — extracts text from images
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [2]:
print(pytesseract.get_tesseract_version())

5.5.0.20241111


In [3]:
# For making HTTP requests to the website
import requests

# For parsing and navigating HTML content (web scraping)
from bs4 import BeautifulSoup

# For writing structured data (image info, captions, OCR) into a CSV file
import csv

# For cleaning and manipulating strings using regular expressions
import re

# For working with images (loading, converting)
from PIL import Image

# For reading binary image data from the web (used by PIL)
from io import BytesIO

# For tensor operations and GPU acceleration (used in deep learning models)
import torch

# Hugging Face Transformers: CLIP model and its processor for vision-language tasks
from transformers import CLIPProcessor, CLIPModel


d:\python3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Load CLIP model
print("Loading CLIP model...")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
print("CLIP model loaded.\n")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading CLIP model...
CLIP model loaded.



In [5]:
# Scrape the website
url = "https://www.magicalmelghat.in"
print(f"🌐 Fetching images from: {url}")
res = requests.get(url)
soup = BeautifulSoup(res.text, "html.parser")

🌐 Fetching images from: https://www.magicalmelghat.in


In [6]:
# Function to clean raw text by removing unwanted characters and extra spaces
def clean_text(text):
    # Remove non-alphanumeric characters (except spaces, commas, and periods)
    cleaned = re.sub(r'[^\w\s,.]', '', text)
    # Remove any HTML-like tags
    cleaned = re.sub(r'<.*?>', '', cleaned)
    # Replace newlines and carriage returns with a space
    cleaned = re.sub(r'[\r\n]+', ' ', cleaned)
    # Replace multiple consecutive spaces with a single space
    cleaned = re.sub(r'\s+', ' ', cleaned)
    # Return the cleaned and stripped text
    return cleaned.strip()

# Function to split long text into individual sentences
def split_into_sentences(text):
    # Split text at the end of periods followed by spaces (basic sentence splitting)
    sentences = re.split(r'(?<=\.)\s*', text)
    # Remove empty strings and strip whitespace from each sentence
    return [s.strip() for s in sentences if s.strip()]

# Function to ensure each text input does not exceed CLIP’s token limit (default 77)
def truncate_by_token_limit(sentences, max_tokens=77):
    truncated_sentences = []
    for sentence in sentences:
        # Tokenize the sentence into subword tokens
        tokens = processor.tokenizer.tokenize(sentence)
        # Trim tokens if they exceed the max limit
        if len(tokens) > max_tokens:
            tokens = tokens[:max_tokens]
        # Convert tokens to input IDs for decoding
        input_ids = processor.tokenizer.convert_tokens_to_ids(tokens)
        # Decode back into a readable string (removes token artifacts like "</w>")
        truncated = processor.tokenizer.decode(input_ids, skip_special_tokens=True)
        truncated_sentences.append(truncated)
    return truncated_sentences

# Function to generate a short version of a caption (default: first 10 words)
def short_caption(text, max_words=10):
    # Split caption into words
    words = text.split()
    # Return first `max_words` words or full text if shorter
    return " ".join(words[:max_words]) if len(words) > max_words else text


In [7]:
# Extract all visible text from the HTML soup (website's raw body content)
body_text = soup.get_text()

# Clean the extracted text using the custom clean_text function
# This removes unnecessary symbols, newlines, and extra whitespace
cleaned_body_text = clean_text(body_text)

# Define any custom base prompts (optional, can be used to steer CLIP captioning)
base_prompts = []

# Combine base prompts with cleaned and split sentences from the website text
# This creates the full list of text inputs to compare with images using CLIP
all_text_inputs = base_prompts + split_into_sentences(cleaned_body_text)

# Ensure all text inputs are within CLIP's token limit (usually 77 tokens max)
split_sentences = truncate_by_token_limit(all_text_inputs, max_tokens=77)

# Print how many processed text prompts are ready for image-text similarity
print(f"Total text inputs for CLIP: {len(split_sentences)}")


Token indices sequence length is longer than the specified maximum sequence length for this model (164 > 77). Running this sequence through the model will result in indexing errors


Total text inputs for CLIP: 79


In [8]:
# Setup CSV
csv_data = [["title", "url", "alt text", "missing alt", "clip caption", "ocr", "ocr_result", "result"]]

In [9]:

# Image scraping
img_tags = soup.find_all("img")
print(f"🔍 Found {len(img_tags)} image(s). Processing...\n")

🔍 Found 105 image(s). Processing...



In [10]:
# Loop through all <img> tags found on the page
for index, img_tag in enumerate(img_tags, start=1):
    img_url = img_tag.get("src")           # Get the image source URL
    alt = img_tag.get("alt")               # Get the alt text, if any

    if not img_url:
        continue                           # Skip if the image URL is missing

    # Ensure the image URL is absolute by appending it to the base URL if needed
    if not img_url.startswith("http"):
        img_url = url + "/" + img_url.lstrip("/")

    print(f"[{index}/{len(img_tags)}] Processing image: {img_url}")

    try:
        # Download and load the image using PIL
        img_response = requests.get(img_url)
        img = Image.open(BytesIO(img_response.content)).convert("RGBA")

        # Use Tesseract OCR to extract any embedded text from the image
        extracted_text = pytesseract.image_to_string(img)
        cleaned_ocr_text = clean_text(extracted_text)  # Clean the OCR output

        # Use CLIP to compare the image with the prepared text prompts
        inputs = processor(
            text=split_sentences,
            images=img,
            return_tensors="pt",
            padding=True,
            truncation=True
        )

        # Disable gradient tracking for inference (faster, no memory tracking)
        with torch.no_grad():
            outputs = model(**inputs)

        # Get the most relevant text prompt (caption) for the image
        logits_per_image = outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)
        caption_index = torch.argmax(probs)
        clip_caption = split_sentences[caption_index]

        # Generate a short caption by trimming the full caption to 5 words
        final_caption = short_caption(clip_caption)

        # Combine OCR text and short caption if OCR was successful
        result_text = f"{cleaned_ocr_text} {final_caption}".strip() if cleaned_ocr_text else final_caption

        # Append the results as a new row in the CSV data
        csv_data.append([
            alt or "",                        # title (from alt text if available)
            img_url,                          # image URL
            alt or "",                        # alt text
            "yes" if not alt else "no",       # flag: missing alt?
            clip_caption,                     # full CLIP-generated caption
            "yes" if cleaned_ocr_text else "no",  # flag: OCR present?
            cleaned_ocr_text,                 # OCR text (if any)
            result_text                       # combined caption/description
        ])

    except Exception as e:
        print(f"Error processing image {img_url}: {e}")  # Log any errors gracefully


[1/105] Processing image: https://www.magicalmelghat.in/public/website/image/jointlogonew.png
[2/105] Processing image: https://www.magicalmelghat.in/public/website/image/melghatlogo-name.png
[3/105] Processing image: https://www.magicalmelghat.in/public/website/image/logo.png
[4/105] Processing image: https://www.magicalmelghat.in/public/website/image/close.png
[5/105] Processing image: https://www.magicalmelghat.in/public/website/image/satymevlogo.png
[6/105] Processing image: https://www.magicalmelghat.in/public/website/image/mh-logo.png
[7/105] Processing image: https://magicalmelghat.in/public/images/banner/1585136941hero1.png
[8/105] Processing image: https://magicalmelghat.in/public/images/banner/1585136982hero2.png
[9/105] Processing image: https://magicalmelghat.in/public/images/banner/1585136983hero3.jpg
[10/105] Processing image: https://www.magicalmelghat.in/public/website/image/meghat-paces/melghat-image1.jpeg
[11/105] Processing image: https://www.magicalmelghat.in/public

In [11]:
# Save to CSV
with open("clip_ocr_report_review_imgs.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerows(csv_data)

print("Done. Data saved to clip_ocr_report_review_imgs.csv")

Done. Data saved to clip_ocr_report_review_imgs.csv
